In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from lightgbm import LGBMClassifier

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "outputs" / "APL_Logistics_ml_ready.csv"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:",PROJECT_ROOT)
print("Data path:",DATA_PATH)
print("Model directory:",MODEL_DIR)
print("Output directory:",OUTPUT_DIR)

Project root: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project
Data path: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs\APL_Logistics_ml_ready.csv
Model directory: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models
Output directory: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs


In [3]:
df = pd.read_csv(DATA_PATH, encoding="latin1")

print("Shape:",df.shape)
df.head()

Shape: (180519, 37)


,Type,Days for shipment (scheduled),Benefit per order,Sales per customer,Category Id,Category Name,Customer City,Customer Country,Customer Segment,Customer State,...,Product Name,Product Price,Shipping Mode,discount_amount_per_unit,sales_per_quantity,profit_per_quantity,price_discount_interaction,scheduled_days_bucket,geo_market_region,Late_delivery_risk
0,DEBIT,4,159.69,472.45,9,Cardio Equipment,Brownsville,EE. UU.,Consumer,TX,...,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class,5.500,99.99,31.938,5.9994,4,Pacific Asia | South Asia,1
1,DEBIT,4,48.71,167.96,29,Shop By Sport,Littleton,EE. UU.,Consumer,CO,...,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,6.398,39.99,9.742,6.3984,4,LATAM | Central America,0
2,DEBIT,4,87.36,181.99,48,Water Sports,Littleton,EE. UU.,Consumer,CO,...,Pelican Sunstream 100 Kayak,199.99,Standard Class,18.000,199.99,87.360,17.9991,4,LATAM | Central America,0
3,DEBIT,4,-41.89,175.99,48,Water Sports,Littleton,EE. UU.,Consumer,CO,...,Pelican Sunstream 100 Kayak,199.99,Standard Class,24.000,199.99,-41.890,23.9988,4,USCA | East of USA,1
4,DEBIT,4,10.00,40.00,24,Women's Apparel,Littleton,EE. UU.,Consumer,CO,...,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,10.000,50.00,10.000,10.0000,4,USCA | East of USA,1


In [4]:
TARGET = "Late_delivery_risk"

print(df[TARGET].value_counts())
print()
print(df[TARGET].value_counts(normalize=True))

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Late_delivery_risk
1    0.548291
0    0.451709
Name: proportion, dtype: float64


In [5]:
x = df.drop(columns=[TARGET].copy())
y = df[TARGET].astype(int).copy()

print("x shape:",x.shape)
print("y shape:",y.shape)

x shape: (180519, 36)
y shape: (180519,)


In [6]:
LEAKAGE_COLUMNS = [
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Delivery Status",
    "Order Status",
    "Delay_Gap"
]

leakage_present = [
    col for col in LEAKAGE_COLUMNS
    if col in x.columns
]

print("Leakage columns still present:")
print(leakage_present)

Leakage columns still present:
[]


In [7]:
TIMING_SENSITIVE_COLUMNS = [
    "Order Profit Per Order",
    "profit_per_quantity",
    "Benefit per order"
]

present_timing_sensitive = [
    col for col in TIMING_SENSITIVE_COLUMNS
    if col in x.columns
]

print("Timing-sensitive columns found:")
print(present_timing_sensitive)

Timing-sensitive columns found:
['Order Profit Per Order', 'profit_per_quantity', 'Benefit per order']


In [8]:
x= x.drop(
    columns = present_timing_sensitive,
    errors= "ignore"
)

print("New x shape:",x.shape)

New x shape: (180519, 33)


In [9]:
numeric_features = x.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = x.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:",len(numeric_features))
print("Numerical features:")
print(numeric_features)

print()

print("Number of categorical features:",len(categorical_features))
print("Categorical features:")
print(categorical_features)

Number of numerical features: 18
Numerical features:
['Days for shipment (scheduled)', 'Sales per customer', 'Category Id', 'Department Id', 'Latitude', 'Longitude', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Product Price', 'discount_amount_per_unit', 'sales_per_quantity', 'price_discount_interaction', 'scheduled_days_bucket']

Number of categorical features: 15
Categorical features:
['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'Order Region', 'Order State', 'Product Name', 'Shipping Mode', 'geo_market_region']


C:\Users\ganesh\AppData\Local\Temp\ipykernel_18420\1658122694.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(


In [10]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:",x_train.shape)
print("Testing data:",x_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training data: (144415, 33)
Testing data: (36104, 33)

Training target distribution:
Late_delivery_risk
1    0.548288
0    0.451712
Name: proportion, dtype: float64

Testing target distribution:
Late_delivery_risk
1    0.548305
0    0.451695
Name: proportion, dtype: float64


In [11]:
numeric_transformer = Pipeline(
    steps = [
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps = [
        ("imputer",SimpleImputer(strategy="most_frequent")),
        (
            "oneshot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=5
            )
        )
    ]
)

In [12]:
preprocessor = ColumnTransformer(
    transformers= [
        ("num",numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [13]:
logistics_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "Classifier",
            LogisticRegression(
                max_iter=150,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

print(logistics_model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Days for shipment '
                                                   '(scheduled)',
                                                   'Sales per customer',
                                                   'Category Id',
                                                   'Department Id', 'Latitude',
                                                   'Longitude',
                                                   'Order Item Discount',
                                                   'Order Item Discoun

In [14]:
print("Training Logistics Regression.....")

logistics_model.fit(x_train,y_train)

print("Logistics Regression training completed.")

Training Logistics Regression.....
Logistics Regression training completed.


c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\.lgvenv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 150 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=150).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
logistics_pred = logistics_model.predict(x_test)
logistics_proba = logistics_model.predict_proba(x_test)[:,1]
print("predictions generated.")

predictions generated.


In [16]:
logistics_accuracy = accuracy_score(y_test, logistics_pred)
logistics_precision = precision_score(y_test, logistics_pred)
logistics_recall = recall_score(y_test, logistics_pred)
logistics_f1 = f1_score(y_test, logistics_pred)
logistics_roc_auc = roc_auc_score(y_test, logistics_proba)
logistics_average_precision = average_precision_score(
    y_test,
    logistics_proba
)

print("Logistics Regression Results")
print("-" *  40)
print("Accuracy:",logistics_accuracy)
print("Precision:",logistics_precision)
print("Recall:",logistics_recall)
print("F1 Score:", logistics_f1)
print("ROC_AUC:", logistics_roc_auc)
print("Average Precision:", logistics_average_precision)

Logistics Regression Results
----------------------------------------
Accuracy: 0.7075116330600487
Precision: 0.8084424258616083
Recall: 0.6114366538694685
F1 Score: 0.6962724344224575
ROC_AUC: 0.7753854058762637
Average Precision: 0.8320056925154069


In [17]:
logistic_cm = confusion_matrix(
    y_test,
    logistics_pred
)

print("Confusion Matrix:")
print(logistic_cm)

Confusion Matrix:
[[13440  2868]
 [ 7692 12104]]


In [18]:
print(
    classification_report(
        y_test,
        logistics_pred,
        target_names=[
            "No Late Risk",
            "Late Risk"
        ]
    )
)

              precision    recall  f1-score   support

No Late Risk       0.64      0.82      0.72     16308
   Late Risk       0.81      0.61      0.70     19796

    accuracy                           0.71     36104
   macro avg       0.72      0.72      0.71     36104
weighted avg       0.73      0.71      0.71     36104



In [19]:
x_train_lgb = x_train.copy()
x_test_lgb = x_test.copy()

for col in categorical_features:
    x_train_lgb[col] = x_train_lgb[col].astype("category")
    
    x_test_lgb[col] = x_test_lgb[col].astype(
        pd.api.types.CategoricalDtype(
            categories = x_train_lgb[col].cat.categories
        )
    )
    
print("LightGBM data prepared.")

LightGBM data prepared.


C:\Users\ganesh\AppData\Local\Temp\ipykernel_18420\4027630800.py:7: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  x_test_lgb[col] = x_test_lgb[col].astype(
C:\Users\ganesh\AppData\Local\Temp\ipykernel_18420\4027630800.py:7: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  x_test_lgb[col] = x_test_lgb[col].astype(


In [20]:
lightgbm_model = LGBMClassifier(
    n_estimators=350,
    learning_rate=0.06,
    num_leaves=31,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print(lightgbm_model)

LGBMClassifier(colsample_bytree=0.9, learning_rate=0.06, n_estimators=350,
               n_jobs=-1, random_state=42, reg_lambda=2.0, subsample=0.9,
               verbosity=-1)


In [21]:
print("Training LightGBM....")

lightgbm_model.fit(
    x_train_lgb,
    y_train,
    categorical_feature=categorical_features
)

print("LightGBM training completed.")

Training LightGBM....
LightGBM training completed.


In [23]:
lightgbm_pred = lightgbm_model.predict(x_test_lgb)

lightgbm_proba = lightgbm_model.predict_proba(
    x_test_lgb
)[:,1]

print("LightGBM prediction generated")

LightGBM prediction generated


In [28]:
lightgbm_accuracy = accuracy_score(
    y_test,
    lightgbm_pred
)

lightgbm_precision = precision_score(
    y_test,
    lightgbm_pred
)

lightgbm_recall = recall_score(
    y_test,
    lightgbm_pred
)

lightgbm_f1 = f1_score(
    y_test,
    lightgbm_pred
)

lightgbm_roc_auc = roc_auc_score(
    y_test,
    lightgbm_proba
)

lightgbm_average_precision = average_precision_score(
    y_test,
    lightgbm_proba
)

print("LightGBM Results:")
print("-" * 40)
print("Accuracy:", lightgbm_accuracy)
print("Precision:", lightgbm_precision)
print("Recall:", lightgbm_recall)
print("F1 Score:", lightgbm_f1)
print("ROC-AUC:", lightgbm_roc_auc)
print("Average Precision:", lightgbm_average_precision)

LightGBM Results:
----------------------------------------
Accuracy: 0.773626191003767
Precision: 0.839913435105574
Recall: 0.7253990705192969
F1 Score: 0.7784674599517524
ROC-AUC: 0.8604106564415959
Average Precision: 0.8955731027710396


In [25]:
lightgbm_cm = confusion_matrix(
    y_test,
    lightgbm_pred
)

print("Confusion Matrix:")
print(lightgbm_cm)

Confusion Matrix:
[[13571  2737]
 [ 5436 14360]]


In [26]:
print(
    classification_report(
        y_test,
        lightgbm_pred,
        target_names= [
            "No Late Risk",
            "Late Risk"
        ]
    )
)

              precision    recall  f1-score   support

No Late Risk       0.71      0.83      0.77     16308
   Late Risk       0.84      0.73      0.78     19796

    accuracy                           0.77     36104
   macro avg       0.78      0.78      0.77     36104
weighted avg       0.78      0.77      0.77     36104



In [30]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "LightGBM"
    ],
    "Accuracy": [
        logistics_accuracy,
        lightgbm_accuracy
    ],
    "Precision": [
        logistics_precision,
        lightgbm_precision
    ],
    "Recall": [
        logistics_recall,
        lightgbm_recall
    ],
    "F1 Score": [
        logistics_f1,
        lightgbm_f1
    ],
    "ROC-AUC": [
        logistics_roc_auc,
        lightgbm_roc_auc
    ],
    "Average Precision": [
        logistics_average_precision,
        lightgbm_average_precision
    ]
})

model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,Logistic Regression,0.707512,0.808442,0.611437,0.696272,0.775385,0.832006
1,LightGBM,0.773626,0.839913,0.725399,0.778467,0.860411,0.895573


In [31]:
model_comparison_rounded = model_comparison.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC",
    "Average Precision"
]

model_comparison_rounded[metric_columns] = (
    model_comparison_rounded[metric_columns]
    .round(4)
)

model_comparison_rounded

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,Logistic Regression,0.7075,0.8084,0.6114,0.6963,0.7754,0.8320
1,LightGBM,0.7736,0.8399,0.7254,0.7785,0.8604,0.8956


In [32]:
comparison_path =  OUTPUT_DIR / "model_comparison.csv"

model_comparison_rounded.to_csv(
    comparison_path,
    index= False
)

print("Saved:", comparison_path)

Saved: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs\model_comparison.csv


In [35]:
logistic_model_path = MODEL_DIR / "logistic_regression_pipeline.joblib"

joblib.dump(
    logistics_model,
    logistic_model_path
)

print("Saved:", logistic_model_path)

Saved: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models\logistic_regression_pipeline.joblib


In [36]:
lightgbm_model_path = MODEL_DIR / "lightgbm_delay_risk_model.joblib"

joblib.dump(
    lightgbm_model,
    lightgbm_model_path
)

print("Saved:", lightgbm_model_path)

Saved: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models\lightgbm_delay_risk_model.joblib


In [37]:
print("Model files:")

for file in MODEL_DIR.iterdir():
    print(file.name)

print("\nOutput files:")

for file in OUTPUT_DIR.iterdir():
    print(file.name)

Model files:
lightgbm_delay_risk_model.joblib
logistic_regression_pipeline.joblib

Output files:
APL_Logistics_cleaned.csv
APL_Logistics_ml_ready.csv
category_summary.csv
country_summary.csv
feature_summary.csv
market_summary.csv
model_comparison.csv
region_summary.csv


In [39]:
print("MODEL TRAINING VALIDATION")
print("=" * 50)

print("Training rows:", len(x_train))
print("Testing rows:", len(x_test))

print("\nModels trained:")
print("- Logistic Regression")
print("- LightGBM")

print("\nComparison:")
display(model_comparison_rounded)

print("\nSaved models:")
print(logistic_model_path)
print(lightgbm_model_path)

print("\nModel training notebook completed successfully.")

MODEL TRAINING VALIDATION
Training rows: 144415
Testing rows: 36104

Models trained:
- Logistic Regression
- LightGBM

Comparison:


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Average Precision
0,Logistic Regression,0.7075,0.8084,0.6114,0.6963,0.7754,0.8320
1,LightGBM,0.7736,0.8399,0.7254,0.7785,0.8604,0.8956



Saved models:
c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models\logistic_regression_pipeline.joblib
c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models\lightgbm_delay_risk_model.joblib

Model training notebook completed successfully.
